# Phase 4.3 exit gate: eager-mode sanity check on real Llama-3.2-1B

Windows the exact head sets built by `build_arms.py` (map25/50/75, derived from the group-level
compressibility map `head_compressibility_group_Llama-3.2-1B.json`) on the real model and checks
PPL cost. `map50` selects the same 64 cheapest KV groups as the map's own recorded `bulk` (K=64)
result, by construction (`group_mask_from_scores` on the same `delta_ppl_group` values) -- so
its PPL should reproduce that recorded number almost exactly. That's the real check: if it
doesn't, something in `build_arms.py`'s ranking/expansion is wrong before we ever touch export.

Run on Colab with a T4 GPU runtime. Needs a HF token with access to the gated
`meta-llama/Llama-3.2-1B` repo.

In [ ]:
!pip install -q transformers datasets scipy huggingface_hub

In [ ]:
from huggingface_hub import login
login()  # paste an HF token with access to meta-llama/Llama-3.2-1B when prompted

In [ ]:
# Pull the arms + source map straight from the public windowed-attention-cpu repo (avoids a
# manual upload step -- both files are already committed there).
!rm -rf /content/windowed-attention-cpu
!git clone --depth 1 https://github.com/parzi-val/windowed-attention-cpu.git /content/windowed-attention-cpu
ARMS_DIR = "/content/windowed-attention-cpu/executorch/phase4_llama"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb, repeat_kv
from datasets import load_dataset
import math, json

# Same eval config as the map-generation notebook (experiment_head_compressibility_llama_3.2_1b.ipynb)
# -- same SEQ_LEN/WINDOW/SINK/EVAL_SEQS/BATCH_SIZE, so this is a like-for-like PPL comparison, not
# a scaled-down approximation. Cheap here because we only do 4 evaluate_ppl() calls (base + 3 arms),
# not the map's own 128-way per-group sweep.
MODEL_ID = "meta-llama/Llama-3.2-1B"
SEQ_LEN = 512
WINDOW = 64
SINK = 4
EVAL_SEQS = 32
BATCH_SIZE = 8

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1. Load Model & Dataset
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, attn_implementation="eager",
                                             torch_dtype=torch.float32).to(device)
model.eval()

print("Loading WikiText-103...")
dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")
encodings = tokenizer("\n\n".join(dataset["text"]), return_tensors="pt")

seqs = []
for i in range(EVAL_SEQS):
    start = i * SEQ_LEN
    seqs.append(encodings.input_ids[0, start:start + SEQ_LEN])
eval_data = torch.stack(seqs)

In [ ]:
# 2. The Surgical Patch (GQA + full RoPE) -- identical to the map-generation notebook, since this
# must reproduce HF exactly (parity gate below) for the PPL numbers to mean anything.

class WindowedLlamaAttention(nn.Module):
    def __init__(self, original_attn, window=64, sink=4):
        super().__init__()
        self.window = window
        self.sink = sink
        self.original_attn = original_attn
        cfg = original_attn.config
        self.num_heads = cfg.num_attention_heads
        self.num_kv_heads = cfg.num_key_value_heads
        self.head_dim = getattr(original_attn, "head_dim", cfg.hidden_size // cfg.num_attention_heads)
        self.n_rep = self.num_heads // self.num_kv_heads
        self.scaling = getattr(original_attn, "scaling", self.head_dim ** -0.5)
        self.mask_heads = torch.zeros(self.num_heads, dtype=torch.bool)

    def forward(self, hidden_states, attention_mask=None, position_ids=None,
                position_embeddings=None, **kwargs):
        bsz, q_len, _ = hidden_states.size()
        hd = self.head_dim
        hidden_shape = (bsz, q_len, -1, hd)

        q = self.original_attn.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        k = self.original_attn.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        v = self.original_attn.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        if position_embeddings is None:
            position_embeddings = kwargs.get("position_embeddings", None)
        if position_ids is None:
            position_ids = torch.arange(q_len, device=hidden_states.device).unsqueeze(0)
        if position_embeddings is not None:
            cos, sin = position_embeddings
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
        else:
            try:
                cos, sin = self.original_attn.rotary_emb(v, position_ids)
            except TypeError:
                cos, sin = self.original_attn.rotary_emb(v, seq_len=q_len)
            q, k = apply_rotary_pos_emb(q, k, cos, sin, position_ids)

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)
        scores = torch.matmul(q, k.transpose(-1, -2)) * self.scaling

        causal = torch.tril(torch.ones(q_len, q_len, device=scores.device)).view(1, 1, q_len, q_len)
        scores = scores.masked_fill(causal == 0, torch.finfo(scores.dtype).min)

        if self.mask_heads.any():
            idx = torch.arange(q_len, device=scores.device)
            valid = (idx[None, :] < self.sink) | (idx[None, :] > (idx[:, None] - self.window))
            valid = valid.view(1, 1, q_len, q_len).expand(bsz, self.num_heads, q_len, q_len)
            head_mask_tensor = self.mask_heads.view(1, self.num_heads, 1, 1).to(scores.device)
            force_mask = head_mask_tensor & (~valid) & (idx[None, :] <= idx[:, None])
            scores = scores.masked_fill(force_mask, torch.finfo(scores.dtype).min)

        probs = F.softmax(scores, dim=-1)
        out = torch.matmul(probs, v)
        out = out.transpose(1, 2).contiguous().view(bsz, q_len, -1)
        return (self.original_attn.o_proj(out), None)


_sample = eval_data[:1].to(device)
with torch.no_grad():
    _ref_logits = model(_sample).logits.float().cpu()

print("Patching Llama layers...")
for layer in model.model.layers:
    layer.self_attn = WindowedLlamaAttention(layer.self_attn, window=WINDOW, sink=SINK)

_sw = getattr(model.model.layers[0].self_attn.original_attn, "sliding_window", None)
assert _sw is None or _sw >= SEQ_LEN, f"active sliding_window={_sw} < SEQ_LEN={SEQ_LEN}: reimpl would diverge from HF"

with torch.no_grad():
    _patched_logits = model(_sample).logits.float().cpu()
_maxdiff = (_ref_logits - _patched_logits).abs().max().item()
print(f"parity gate: max |logit_patched - logit_HF| = {_maxdiff:.2e}")
assert _maxdiff < 1e-2, "PATCH MISMATCH -- windowed GQA attention does not reproduce HF; do not trust the result."
print("PASS: monkeypatch reproduces HF exactly.")

In [ ]:
def evaluate_ppl():
    tot_loss, tot_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(0, EVAL_SEQS, BATCH_SIZE):
            batch = eval_data[i:i + BATCH_SIZE].to(device)
            outputs = model(batch, labels=batch)
            tot_loss += outputs.loss.item() * batch.numel()
            tot_tokens += batch.numel()
    return math.exp(tot_loss / tot_tokens)

def set_head_mask_from_arm(head_windowed):
    for l, layer in enumerate(model.model.layers):
        layer.self_attn.mask_heads = torch.tensor(head_windowed[l], dtype=torch.bool)

set_head_mask_from_arm([[False] * model.config.num_attention_heads] * model.config.num_hidden_layers)
print("Running baseline...")
base_ppl = evaluate_ppl()
print(f"Baseline PPL: {base_ppl:.4f}")

In [ ]:
# 3. Evaluate each arm from build_arms.py's output
arms_data = json.load(open(f"{ARMS_DIR}/llama_3.2_1b_arms.json"))
map_json = json.load(open(f"{ARMS_DIR}/head_compressibility_group_Llama-3.2-1B.json"))

results = {"base_ppl": base_ppl, "base_ppl_recorded": map_json["base_ppl"], "arms": {}}
print(f"base_ppl this run: {base_ppl:.4f}  vs. recorded in map: {map_json['base_ppl']:.4f}\n")

for key in ["map25", "map50", "map75"]:
    hw = arms_data["arms"][key]["head_windowed"]
    set_head_mask_from_arm(hw)
    ppl = evaluate_ppl()
    results["arms"][key] = ppl
    print(f"{key}: PPL = {ppl:.4f}  (delta = +{ppl - base_ppl:.4f})")

set_head_mask_from_arm([[False] * model.config.num_attention_heads] * model.config.num_hidden_layers)

print(f"\nRecorded bulk (K=64, i.e. map50-equivalent) in the source map:")
print(f"  base={map_json['bulk']['base']:.4f}  map_guided={map_json['bulk']['map_guided']:.4f}  "
      f"random={map_json['bulk']['random']:.4f}")
print(f"\nThis run's map50 ({results['arms']['map50']:.4f}) should closely match the recorded "
      f"map_guided ({map_json['bulk']['map_guided']:.4f}) -- both select the same 64 cheapest groups.")

monotonic = base_ppl <= results["arms"]["map25"] <= results["arms"]["map50"] <= results["arms"]["map75"]
print(f"\nMonotonic base <= map25 <= map50 <= map75: {monotonic}")

In [ ]:
json.dump(results, open("/content/phase4_3_sanity_results.json", "w"), indent=1)
print("saved /content/phase4_3_sanity_results.json -- download this and drop it into \n"
      "windowed-attention-cpu/executorch/phase4_llama/ before committing.")
from google.colab import files
files.download("/content/phase4_3_sanity_results.json")